# Retrieval Experiments: TF-IDF vs Dense Retrieval

## 1. Цель этапа

В предыдущем этапе была построена и проверена knowledge base:

- clinical guidelines преобразованы в retrieval chunks;
- для chunks рассчитаны dense embeddings с `BAAI/bge-base-en-v1.5`;
- создан frozen retrieval benchmark из 47 вопросов.

На этом этапе сравниваются разные методы retrieval на одних и тех же:

- chunks;
- evaluation queries;
- gold-relevance annotations.

Основной baseline:

1. TF-IDF — lexical retrieval;
2. BGE — dense semantic retrieval.

Если анализ ошибок покажет, что методы дополняют друг друга,
дополнительно будет рассмотрен hybrid retrieval.

`test` dataset на этом этапе не используется.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

import torch
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)

In [2]:
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


RETRIEVAL_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "retrieval"
)

MATERIALS_DIR = (
    PROJECT_ROOT
    / "materials"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
from src.retrieval import (
    load_retrieval_config,
    load_retrieval_data,
    load_embedding_model,
    dense_search,

)

In [4]:
retrieval_config = load_retrieval_config(
    RETRIEVAL_DATA_DIR
)

chunks_df, document_embeddings = (
    load_retrieval_data(
        RETRIEVAL_DATA_DIR
    )
)

print(
    "Chunks:",
    len(chunks_df),
)

print(
    "Dense embeddings:",
    document_embeddings.shape,
)

retrieval_config

Chunks: 2135
Dense embeddings: (2135, 768)


{'embedding_model': 'BAAI/bge-base-en-v1.5',
 'query_prefix': 'Represent this sentence for searching relevant passages: ',
 'target_chunk_tokens': 384,
 'chunk_overlap_tokens': 64,
 'normalize_embeddings': True,
 'embedding_dimension': 768,
 'num_chunks': 2135}

In [5]:
RETRIEVAL_EVAL_PATH = (
    MATERIALS_DIR
    / "retrieval_eval_v1.jsonl"
)

retrieval_eval_df = pd.read_json(
    RETRIEVAL_EVAL_PATH,
    lines=True,
)

print(
    "Evaluation questions:",
    len(retrieval_eval_df),
)

print()

print(
    retrieval_eval_df[
        "question_source"
    ].value_counts()
)

retrieval_eval_df.head()

Evaluation questions: 47

question_source
guideline    40
real_dev      7
Name: count, dtype: int64


,eval_id,candidate_id,question_source,query,relevant_document_id,relevant_sections
0,dev_0005,candidate_0005,real_dev,Hi my boyfriend and I have been together for o...,va_dod_major_depression_2022,[VIII. Algorithm > A. Module A: Initial Assess...
1,dev_0014,candidate_0014,real_dev,"Hi, I am suffering from lower back pain going ...",va_dod_low_back_pain_2022,[IX. Recommendations > A. Evaluation and Diagn...
2,dev_0029,candidate_0029,real_dev,I have been told that I have stage 5 advanced ...,va_dod_ckd_2025,"[IX. Recommendations, VIII. Algorithm, Appendi..."
3,dev_0048,candidate_0048,real_dev,hi sir iam having a sciatica problem and lumbe...,va_dod_low_back_pain_2022,"[IX. Recommendations > D. Pharmacotherapy, IX...."
4,dev_0078,candidate_0078,real_dev,two months now into it...started off lower bac...,va_dod_low_back_pain_2022,[IX. Recommendations > A. Evaluation and Diagn...


## 2. TF-IDF retrieval

Dense retrieval ищет семантически похожие тексты.

TF-IDF использует другой принцип: он оценивает совпадение слов и фраз
между query и document chunks.

Например, запрос:

`high potassium in chronic kidney disease`

может хорошо находить текст, содержащий те же или близкие lexical terms.

TF-IDF используется как простой sparse baseline, чтобы проверить,
действительно ли semantic embeddings дают преимущество по сравнению
с обычным lexical retrieval.

Для честного сравнения TF-IDF индексируется по тем же chunks,
которые использовались для dense retrieval.

In [6]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    norm="l2",
    dtype=np.float32,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    chunks_df["embedding_text"]
)

print(
    "TF-IDF matrix:",
    tfidf_matrix.shape,
)

print(
    "Vocabulary size:",
    len(tfidf_vectorizer.vocabulary_),
)

TF-IDF matrix: (2135, 129847)
Vocabulary size: 129847


In [7]:
def tfidf_search(
    query,
    top_k=5,
):
    query_vector = (
        tfidf_vectorizer.transform(
            [query]
        )
    )

    scores = (
        tfidf_matrix
        @ query_vector.T
    )

    scores = (
        scores
        .toarray()
        .ravel()
    )

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = (
        chunks_df
        .iloc[top_indices]
        .copy()
    )

    results.insert(
        0,
        "score",
        scores[top_indices],
    )

    return (
        results[
            [
                "score",
                "chunk_id",
                "document_id",
                "section_path",
                "page",
                "pdf_page",
                "text",
            ]
        ]
        .reset_index(drop=True)
    )

In [8]:
query = (
    "How should hyperkalemia be managed "
    "in patients with chronic kidney disease?"
)

tfidf_results = tfidf_search(
    query,
    top_k=5,
)

pd.set_option(
    "display.max_colwidth",
    300,
)

tfidf_results

,score,chunk_id,document_id,section_path,page,pdf_page,text
0,0.215331,record_00760_chunk_01,va_dod_ckd_2025,IX. Recommendations,34,34,"disease not on dialysis, we recommend the initiation of statins to reduce major adverse cardiovascular events and mortality. Strong for Reviewed, New-replaced Other Medications to Decrease Cardiovascular Disease and Kidney Outcomes 20. In patients with autosomal dominant polycystic kidney diseas..."
1,0.189164,record_00760_chunk_00,va_dod_ckd_2025,IX. Recommendations,34,34,"Sub-topic Topic # Recommendation Strengtha Categoryb and continuing sodium-glucose co-transporter 2 inhibitors until start of dialysis. Strong for Reviewed, New-replaced 16. We recommend adding a glucagon-like peptide-1 receptor agonist to an angiotensin-converting enzyme inhibitor or angiotensi..."
2,0.177371,record_00759_chunk_01,va_dod_ckd_2025,IX. Recommendations,33,33,"a thiazide diuretic or calcium channel blocker to reduce blood pressure in patients with chronic kidney disease and hypertension not controlled on an angiotensin-converting enzyme inhibitor or angiotensin II receptor blocker. Weak for Reviewed, New-added 14. In patients with advanced chronic kid..."
3,0.163285,record_00795_chunk_00,va_dod_ckd_2025,IX. Recommendations,69,69,"primary therapy available for patients with rapidly progressive ADPKD. Finally, it must be emphasized that the benefits reported in slowing CKD progression remain greatest in early-stage CKD, highlighting the need for early nephrology subspecialty referral for evaluating the safe and appropriate..."
4,0.162268,record_00759_chunk_00,va_dod_ckd_2025,IX. Recommendations,33,33,"Sub-topic Topic # Recommendation Strengtha Categoryb Weak for Not reviewed, Not changed 8. We suggest utilizing shared decision-making regarding kidney replacement therapy versus conservative management. Weak for Not reviewed, Amended 9. In patients with high co-occurring conditions/low function..."


## 3. Оценка TF-IDF retrieval

TF-IDF оценивается на том же frozen benchmark, который использовался
для dense retrieval.

Gold-разметка не изменяется.

Основные метрики:

- Hit@1
- Hit@3
- Hit@5
- MRR@10

Метрики считаются отдельно для `guideline` и `real_dev`.

In [9]:
def evaluate_tfidf_query(
    row,
    top_k=10,
):
    results = tfidf_search(
        row["query"],
        top_k=top_k,
    )

    gold_document = (
        row["relevant_document_id"]
    )

    gold_sections = set(
        row["relevant_sections"]
    )

    is_relevant = (
        results["document_id"].eq(
            gold_document
        )
        &
        results["section_path"].isin(
            gold_sections
        )
    )

    positions = np.flatnonzero(
        is_relevant.to_numpy()
    )

    first_relevant_rank = (
        int(positions[0] + 1)
        if len(positions) > 0
        else None
    )

    return {
        "eval_id": row["eval_id"],
        "question_source": (
            row["question_source"]
        ),
        "first_relevant_rank": (
            first_relevant_rank
        ),
        "hit_at_1": (
            first_relevant_rank is not None
            and first_relevant_rank <= 1
        ),
        "hit_at_3": (
            first_relevant_rank is not None
            and first_relevant_rank <= 3
        ),
        "hit_at_5": (
            first_relevant_rank is not None
            and first_relevant_rank <= 5
        ),
        "rr_at_10": (
            1 / first_relevant_rank
            if first_relevant_rank is not None
            else 0.0
        ),
        "top1_document_id": (
            results.iloc[0][
                "document_id"
            ]
        ),
        "top1_section_path": (
            results.iloc[0][
                "section_path"
            ]
        ),
    }

In [10]:
tfidf_eval_results = []

for _, row in retrieval_eval_df.iterrows():
    tfidf_eval_results.append(
        evaluate_tfidf_query(
            row,
            top_k=10,
        )
    )

tfidf_eval_results_df = pd.DataFrame(
    tfidf_eval_results
)

tfidf_eval_results_df.head()

,eval_id,question_source,first_relevant_rank,hit_at_1,hit_at_3,hit_at_5,rr_at_10,top1_document_id,top1_section_path
0,dev_0005,real_dev,NaN,False,False,False,0.0,cdc_sti_2021,"Diseases Characterized by Vulvovaginal Itching, Burning, Irritation, Odor, or Discharge > Trichomoniasis > Recurrent Trichomoniasis"
1,dev_0014,real_dev,2.0,False,True,True,0.5,va_dod_asthma_2025,IX. Recommendations > B. Treatment and Management
2,dev_0029,real_dev,1.0,True,True,True,1.0,va_dod_ckd_2025,IX. Recommendations
3,dev_0048,real_dev,1.0,True,True,True,1.0,va_dod_low_back_pain_2022,IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy
4,dev_0078,real_dev,1.0,True,True,True,1.0,va_dod_low_back_pain_2022,VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain


In [11]:
def summarize_retrieval_metrics(df):
    return pd.Series(
        {
            "n": len(df),
            "Hit@1": (
                df["hit_at_1"].mean()
            ),
            "Hit@3": (
                df["hit_at_3"].mean()
            ),
            "Hit@5": (
                df["hit_at_5"].mean()
            ),
            "MRR@10": (
                df["rr_at_10"].mean()
            ),
        }
    )

In [12]:
tfidf_overall_metrics = (
    summarize_retrieval_metrics(
        tfidf_eval_results_df
    )
)

tfidf_overall_metrics

n         47.000000
Hit@1      0.340426
Hit@3      0.595745
Hit@5      0.702128
MRR@10     0.509304
dtype: float64

In [ ]:
tfidf_metrics_by_source = (
    tfidf_eval_results_df
    .groupby("question_source")
    .apply(
        summarize_retrieval_metrics
    )
)

tfidf_metrics_by_source

In [14]:
tfidf_doc_hit_at_1 = (
    tfidf_eval_results_df[
        "top1_document_id"
    ].to_numpy()
    ==
    retrieval_eval_df[
        "relevant_document_id"
    ].to_numpy()
).mean()

print(
    "TF-IDF Document Hit@1:",
    tfidf_doc_hit_at_1,
)

TF-IDF Document Hit@1: 0.9361702127659575


## 4. Сравнение TF-IDF и dense retrieval

TF-IDF и BGE были оценены на одном frozen benchmark и по одной
section-level gold-разметке.

Dense retrieval существенно превосходит TF-IDF по Hit@k и MRR.

Следующий вопрос — есть ли запросы, на которых TF-IDF находит релевантный
раздел лучше BGE. Если такие случаи существуют, комбинация lexical и
semantic retrieval может быть полезна для hybrid retrieval.

In [15]:
dense_eval_results_df = pd.read_csv(
    RESULTS_DIR / "retrieval_dense_eval_v1.csv"
)

dense_eval_results_df.head()

,eval_id,candidate_id,question_source,query,first_relevant_rank,hit_at_1,hit_at_3,hit_at_5,rr_at_10,top1_document_id,top1_section_path,top1_score
0,dev_0005,candidate_0005,real_dev,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stre...,8.0,False,False,False,0.125,va_dod_major_depression_2022,II. Background > A. Description of Major Depressive Disorder (MDD),0.628073
1,dev_0014,candidate_0014,real_dev,"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to deter...",1.0,True,True,True,1.000,va_dod_low_back_pain_2022,IX. Recommendations > A. Evaluation and Diagnostic Approach,0.668335
2,dev_0029,candidate_0029,real_dev,"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be...",1.0,True,True,True,1.000,va_dod_ckd_2025,IX. Recommendations,0.723086
3,dev_0048,candidate_0048,real_dev,hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but ...,1.0,True,True,True,1.000,va_dod_low_back_pain_2022,IX. Recommendations > D. Pharmacotherapy,0.692542
4,dev_0078,candidate_0078,real_dev,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for an...",2.0,False,True,True,0.500,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,0.646642


In [16]:
comparison_df = (
    dense_eval_results_df[
        [
            "eval_id",
            "question_source",
            "first_relevant_rank",
        ]
    ]
    .rename(
        columns={
            "first_relevant_rank": "dense_rank"
        }
    )
    .merge(
        tfidf_eval_results_df[
            [
                "eval_id",
                "first_relevant_rank",
            ]
        ].rename(
            columns={
                "first_relevant_rank": "tfidf_rank"
            }
        ),
        on="eval_id",
        how="inner",
    )
)

comparison_df.head()

,eval_id,question_source,dense_rank,tfidf_rank
0,dev_0005,real_dev,8.0,NaN
1,dev_0014,real_dev,1.0,2.0
2,dev_0029,real_dev,1.0,1.0
3,dev_0048,real_dev,1.0,1.0
4,dev_0078,real_dev,2.0,1.0


In [17]:
comparison_df["dense_rank_cmp"] = (
    comparison_df["dense_rank"]
    .fillna(np.inf)
)

comparison_df["tfidf_rank_cmp"] = (
    comparison_df["tfidf_rank"]
    .fillna(np.inf)
)

comparison_df["better_method"] = np.select(
    [
        comparison_df["dense_rank_cmp"]
        < comparison_df["tfidf_rank_cmp"],

        comparison_df["tfidf_rank_cmp"]
        < comparison_df["dense_rank_cmp"],
    ],
    [
        "dense",
        "tfidf",
    ],
    default="tie",
)

comparison_df["better_method"].value_counts()

better_method
dense    28
tie      14
tfidf     5
Name: count, dtype: int64

In [18]:
tfidf_better_df = (
    comparison_df[
        comparison_df["better_method"].eq("tfidf")
    ]
    .merge(
        retrieval_eval_df[
            [
                "eval_id",
                "query",
                "relevant_document_id",
                "relevant_sections",
            ]
        ],
        on="eval_id",
        how="left",
    )
    .sort_values(
        [
            "tfidf_rank_cmp",
            "dense_rank_cmp",
        ]
    )
)

tfidf_better_df[
    [
        "eval_id",
        "question_source",
        "query",
        "dense_rank",
        "tfidf_rank",
    ]
]

,eval_id,question_source,query,dense_rank,tfidf_rank
0,dev_0078,real_dev,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for an...",2.0,1.0
1,ckd_01,guideline,"How should chronic kidney disease be evaluated, staged, and monitored over time?",2.0,1.0
3,lbp_01,guideline,How should a patient with low back pain be evaluated to determine whether further diagnostic testing is needed?,2.0,1.0
4,lbp_05,guideline,What non-surgical invasive treatments may be considered for patients with persistent low back pain?,3.0,1.0
2,asthma_01,guideline,How should asthma be diagnosed and assessed in a patient with respiratory symptoms?,NaN,6.0


In [19]:
tfidf_rescues_df = comparison_df[
    comparison_df["dense_rank"].isna()
    & comparison_df["tfidf_rank"].notna()
]

tfidf_rescues_df

,eval_id,question_source,dense_rank,tfidf_rank,dense_rank_cmp,tfidf_rank_cmp,better_method
22,asthma_01,guideline,NaN,6.0,inf,6.0,tfidf


### Анализ случаев, где TF-IDF имеет лучший strict rank

TF-IDF получил более высокий strict section-level rank для нескольких запросов.

Это ещё не означает, что dense retrieval вернул нерелевантный результат:
section-level gold может быть неполным.

Поэтому сравниваем top-ranked chunks обоих методов вручную,
прежде чем решать, нужен ли hybrid retrieval.

In [20]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embedding_model = load_embedding_model(
    model_name=retrieval_config["embedding_model"],
    device=device,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7960.43it/s]


In [21]:
def compare_retrieval_case(eval_id, top_k=3):
    row = retrieval_eval_df[
        retrieval_eval_df["eval_id"].eq(eval_id)
    ].iloc[0]

    print("=" * 100)
    print("EVAL ID:", eval_id)

    print("\nQUERY:")
    print(row["query"])

    print("\nGOLD SECTIONS:")
    for section in row["relevant_sections"]:
        print("-", section)

    dense_results = dense_search(
        query=row["query"],
        chunks_df=chunks_df,
        document_embeddings=document_embeddings,
        embedding_model=embedding_model,
        query_prefix=retrieval_config["query_prefix"],
        top_k=top_k,
    )

    tfidf_results = tfidf_search(
        row["query"],
        top_k=top_k,
    )

    print("\nDENSE:")
    display(
        dense_results[
            [
                "score",
                "document_id",
                "section_path",
                "text",
            ]
        ]
    )

    print("\nTF-IDF:")
    display(
        tfidf_results[
            [
                "score",
                "document_id",
                "section_path",
                "text",
            ]
        ]
    )

In [22]:
for eval_id in [
    "dev_0078",
    "ckd_01",
    "lbp_01",
    "lbp_05",
]:
    compare_retrieval_case(
        eval_id,
        top_k=3,
    )

EVAL ID: dev_0078

QUERY:
two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable.

GOLD SECTIONS:
- IX. Recommendations > A. Evaluation and Diagnostic Approach
- VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain
- VIII. Algorithm > B. Module B: Management of Low Back Pain

DENSE:


,score,document_id,section_path,text
0,0.646642,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,"LBP has been defined as pain, muscle tension, or stiffness localized below the costal margin and above the inferior gluteal folds with or without leg symptoms.(2) Categorizations defined by duration vary but are often delineated as acute (less than four weeks), subacute (4 – 12 weeks), or chroni..."
1,0.644570,va_dod_low_back_pain_2022,VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain,"Sidebar 2: Evaluation for Possible Other Conditionsa Possible Other Conditions Red Flags (e.g., signs, symptoms, history) Suggested Evaluationb • Radicular back pain (e.g., sciatica) • Lower extremity dysesthesia and/or paresthesia None Herniated disc • Severe/progressive lower extremity neurolo..."
2,0.614048,va_dod_low_back_pain_2022,Appendix F: Glossary,"Category Term Definition Acute LBP LBP present for fewer than four weeks. Sometimes grouped with subacute LBP as symptoms present for fewer than 12 weeks. Cauda equina syndrome Compression on nerve roots in the lumbosacral spine, usually due to a massive, centrally herniated disc, or severe lumb..."



TF-IDF:


,score,document_id,section_path,text
0,0.134651,va_dod_low_back_pain_2022,VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain,Abbreviation: LBP: low back pain
1,0.118369,va_dod_low_back_pain_2022,II. Background > B. Epidemiology and Impact,"In a study of U.S. healthcare costs from 1996 through 2016, spending related to LBP and neck pain was the highest out of 154 conditions. In 2016, the estimated spending related to LBP and neck pain was $134.5 billion.(10) LBP and neurogenic claudication can be caused by degenerative lumbar spina..."
2,0.105913,va_dod_low_back_pain_2022,VIII. Algorithm > B. Module B: Management of Low Back Pain,a See the VA/DoD Clinical Practice Guideline for the Use of Opioids in the Management of Chronic Pain. Available at: https://www.healthquality.va.gov/. Abbreviation: CPG: clinical practice guideline; DoD: Department of Defense; LBP: low back pain; VA: Department of Veterans Affairs


EVAL ID: ckd_01

QUERY:
How should chronic kidney disease be evaluated, staged, and monitored over time?

GOLD SECTIONS:
- VIII. Algorithm
- Appendix G. Alternative Text Descriptions of Algorithms
- Appendix I. Monitoring of CKD Table

DENSE:


...

,score,document_id,section_path,text
0,0.204824,va_dod_low_back_pain_2022,VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain,Abbreviation: LBP: low back pain
1,0.171071,va_dod_low_back_pain_2022,IX. Recommendations,"Topic # Recommendation Strengtha Categoryb 1. Strong for Reviewed, Amended For patients with low back pain, we recommend the history and physical examination include evaluation for progressive or otherwise serious neurologic deficits and other red flags (e.g., signs, symptoms, history) associate..."
2,0.167534,va_dod_low_back_pain_2022,IX. Recommendations,"Topic # Recommendation Strengtha Categoryb 31. For patients with chronic low back pain, we suggest lumbar medial branch and/or sacral lateral branch radiofrequency ablation. Weak for Reviewed, New-replaced 32. For patients with low back pain, there is insufficient evidence to recommend for or ag..."


EVAL ID: lbp_05

QUERY:
What non-surgical invasive treatments may be considered for patients with persistent low back pain?

GOLD SECTIONS:
- IX. Recommendations > F. Non-surgical Invasive Therapy

DENSE:


,score,document_id,section_path,text
0,0.727112,va_dod_low_back_pain_2022,IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy,manipulation results in modest but clinically important reductions in pain and disability in patients with chronic LBP.(121-128) Rubinstein et al. (2019) compared spinal manipulative therapy (SMT) to recommended and non-recommended treatments.(128) A treatment was considered recommended or non-r...
1,0.723789,va_dod_low_back_pain_2022,IX. Recommendations > C. Non-pharmacologic and Non-invasive Therapy,"activity prescribed by a clinician to improve pain, disability, and physical function. This includes exercise programs targeted at the lumbar, abdominal, and hip muscles (often referred to as the “core”) and generalized exercises not specifically targeting the back (e.g., aerobic training on a b..."
2,0.714820,va_dod_low_back_pain_2022,IX. Recommendations > F. Non-surgical Invasive Therapy,"of this evidence review. There is some variability in patient preferences regarding this treatment. While some patients seek out emerging interventional therapies, ortho-biologic injections for LBP are infrequently requested. As is the case with invasive procedures, some patients who are needle-..."



TF-IDF:


,score,document_id,section_path,text
0,0.220748,va_dod_low_back_pain_2022,IX. Recommendations > F. Non-surgical Invasive Therapy,"study on RFA in the 2017 VA/DoD LBP CPG found no y Recommendations for “patients with low back pain” encompass patient populations with acute, subacute, or chronic LBP with or without neurological symptoms. z Recommendations for “patients with low back pain” encompass patient populations with ac..."
1,0.214650,va_dod_low_back_pain_2022,IX. Recommendations,"Topic # Recommendation Strengtha Categoryb 31. For patients with chronic low back pain, we suggest lumbar medial branch and/or sacral lateral branch radiofrequency ablation. Weak for Reviewed, New-replaced 32. For patients with low back pain, there is insufficient evidence to recommend for or ag..."
2,0.206478,va_dod_low_back_pain_2022,VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain,Abbreviation: LBP: low back pain


## 5. Итог сравнения TF-IDF и dense retrieval

TF-IDF использовался как lexical baseline и сравнивался с BGE dense retrieval
на одном frozen benchmark из 47 вопросов.

Результаты:

| Metric | TF-IDF | BGE dense |
|---|---:|---:|
| Hit@1 | 0.340 | 0.830 |
| Hit@3 | 0.596 | 0.936 |
| Hit@5 | 0.702 | 0.936 |
| MRR@10 | 0.509 | 0.882 |
| Document Hit@1 | 0.936 | 1.000 |

Dense retrieval существенно превосходит TF-IDF по section-level ranking.

TF-IDF хорошо определяет тематически подходящий документ, но часто высоко
ранжирует chunks из нужного section, которые сами по себе содержат мало
полезной информации. Например, для некоторых low-back-pain queries top-ranked
TF-IDF chunk содержал только расшифровку аббревиатуры `LBP`.

Ручной анализ случаев, где TF-IDF имел лучший strict rank, показал, что
большинство таких преимуществ связано с неполнотой section-level gold или
с формальным попаданием малоинформативного chunk в gold-section.

Явное содержательное преимущество TF-IDF было обнаружено для запроса о
non-surgical invasive treatment of low back pain, где lexical matching
позволил лучше различить `invasive` и `non-invasive` therapy.

Однако это единичный случай, тогда как BGE существенно лучше на benchmark
в целом. Поэтому добавление hybrid retrieval на данном этапе не обосновано.

Для дальнейшего RAG используется BGE dense retrieval.

In [23]:
tfidf_eval_results_df.to_csv(
    RESULTS_DIR / "retrieval_tfidf_eval_v1.csv",
    index=False,
)

comparison_df.to_csv(
    RESULTS_DIR / "retrieval_tfidf_vs_dense_v1.csv",
    index=False,
)

In [24]:
retrieval_method_comparison_df = pd.DataFrame(
    [
        {
            "method": "TF-IDF",
            "Hit@1": 0.340426,
            "Hit@3": 0.595745,
            "Hit@5": 0.702128,
            "MRR@10": 0.509304,
            "Document_Hit@1": 0.936170,
        },
        {
            "method": "BGE dense",
            "Hit@1": 0.829787,
            "Hit@3": 0.936170,
            "Hit@5": 0.936170,
            "MRR@10": 0.882092,
            "Document_Hit@1": 1.0,
        },
    ]
)

retrieval_method_comparison_df.to_csv(
    RESULTS_DIR / "retrieval_method_comparison_v1.csv",
    index=False,
)

retrieval_method_comparison_df

,method,Hit@1,Hit@3,Hit@5,MRR@10,Document_Hit@1
0,TF-IDF,0.340426,0.595745,0.702128,0.509304,0.93617
1,BGE dense,0.829787,0.936170,0.936170,0.882092,1.00000


## 6. FAISS как альтернативный механизм поиска по BGE-векторам

Текущий dense retrieval уже использует BGE embeddings и cosine similarity:

query
↓
BGE embedding
↓
сравнение с embeddings всех chunks
↓
сортировка
↓
top-k.

В текущей реализации сходство считается напрямую через умножение матриц NumPy.

FAISS не заменяет BGE и не создаёт новые embeddings.
Он заменяет только механизм поиска ближайших векторов.

Поскольку embeddings документов и запросов L2-нормализованы,
cosine similarity равна inner product.

Поэтому для точного сравнения используем:

`faiss.IndexFlatIP`

Схема становится:

query
↓
BGE embedding
↓
FAISS IndexFlatIP
↓
top-k.

Важно:

`IndexFlatIP` выполняет точный поиск, поэтому при одинаковых нормализованных
embeddings ожидается практически тот же порядок результатов, что и у
текущего NumPy-поиска.

Следовательно, на этом этапе мы проверяем не новый способ представления текста,
а альтернативный поисковый механизм.

In [25]:
try:
    import faiss
except ImportError as exc:
    raise ImportError(
        "Не найден пакет FAISS. "
        "Установи зависимость `faiss-cpu` и перезапусти ядро."
    ) from exc

In [26]:
faiss_embeddings = np.ascontiguousarray(
    document_embeddings.astype(
        np.float32
    )
)

embedding_norms = np.linalg.norm(
    faiss_embeddings,
    axis=1,
)

print(
    "Минимальная норма embedding:",
    round(
        float(
            embedding_norms.min()
        ),
        6,
    ),
)

print(
    "Максимальная норма embedding:",
    round(
        float(
            embedding_norms.max()
        ),
        6,
    ),
)

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-3,
), (
    "Embeddings документов не L2-нормализованы. "
    "Нельзя напрямую интерпретировать inner product как cosine similarity."
)

Минимальная норма embedding: 1.0
Максимальная норма embedding: 1.0


In [27]:
FAISS_DIMENSION = (
    faiss_embeddings.shape[1]
)

faiss_index = faiss.IndexFlatIP(
    FAISS_DIMENSION
)

faiss_index.add(
    faiss_embeddings
)

print(
    "Размерность векторов:",
    FAISS_DIMENSION,
)

print(
    "Векторов в FAISS:",
    faiss_index.ntotal,
)

Размерность векторов: 768
Векторов в FAISS: 2135


In [28]:
def encode_bge_query(
    query,
):
    query_text = (
        retrieval_config[
            "query_prefix"
        ]
        + query
    )

    query_embedding = (
        embedding_model.encode(
            query_text,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
    )

    return query_embedding

In [29]:
def faiss_search(
    query,
    top_k=5,
):
    query_embedding = (
        encode_bge_query(
            query
        )
        .reshape(1, -1)
    )

    scores, indices = (
        faiss_index.search(
            query_embedding,
            top_k,
        )
    )

    scores = scores[0]
    indices = indices[0]

    results = (
        chunks_df
        .iloc[indices]
        .copy()
    )

    results.insert(
        0,
        "score",
        scores,
    )

    return (
        results[
            [
                "score",
                "chunk_id",
                "document_id",
                "section_path",
                "page",
                "pdf_page",
                "text",
            ]
        ]
        .reset_index(drop=True)
    )

In [30]:
faiss_example = faiss_search(
    query,
    top_k=5,
)

faiss_example

,score,chunk_id,document_id,section_path,page,pdf_page,text
0,0.798135,record_00834_chunk_01,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > D. Dietary Considerations,147,147,"for the Evaluation and Management of Chronic Kidney Disease (3) recommends, “Provide advice to limit the intake of foods rich in bioavailable potassium (e.g., processed foods) for people with CKD G3-G5 who have a history of hyperkalemia.” The involvement of a renal dietitian can be helpful, as c..."
1,0.793694,record_00832_chunk_01,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,146,146,upper limit of normal range They acknowledge that it is not known whether EKG changes are sensitive in the prediction of potentially lethal arrhythmia. The KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease (3) recommends the following steps to man...
2,0.789073,record_00832_chunk_00,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,146,146,"The aggressiveness of treatment of hyperkalemia depends on the degree of elevation and the presence or absence of electrocardiogram (EKG) findings. Observationally, the risk of death from a given level of hyperkalemia is lower in more advanced CKD, suggesting that there are adaptive mechanisms t..."
3,0.768300,record_00831_chunk_00,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > B. Definition,146,146,"An acute episode of hyperkalemia is a potassium result above the upper limit of normal that is not known to be chronic. However, there is no consensus on the magnitude, duration, and frequency of elevated potassium values that define chronicity.(277)"
4,0.762154,record_00833_chunk_00,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,147,147,• Optimize serum bicarbonate levels • Consider potassium-exchange agents Third-line (Last resort): • Reduce or discontinue RAASi/MRA; restart in future if patient condition allows Providers may consider reducing or stopping ACEI/ARB when eGFR drops below a given eGFR threshold or when hyperkalem...


## 7. Сравнение текущего dense-поиска и FAISS

Для честного сравнения оба варианта используют:

- одну и ту же модель `BAAI/bge-base-en-v1.5`;
- одни и те же embeddings документов;
- одинаковую нормализацию;
- одинаковые запросы;
- одинаковый retrieval benchmark.

Различается только способ поиска:

1. NumPy: прямое вычисление `document_embeddings @ query_embedding`;
2. FAISS: точный поиск через `IndexFlatIP`.

Если реализация корректна, retrieval-метрики должны совпадать или отличаться
только из-за редких случаев одинаковых scores.

In [31]:
def evaluate_faiss_query(
    row,
    top_k=10,
):
    results = faiss_search(
        row["query"],
        top_k=top_k,
    )

    gold_document = (
        row[
            "relevant_document_id"
        ]
    )

    gold_sections = set(
        row[
            "relevant_sections"
        ]
    )

    is_relevant = (
        results[
            "document_id"
        ].eq(
            gold_document
        )
        &
        results[
            "section_path"
        ].isin(
            gold_sections
        )
    )

    positions = np.flatnonzero(
        is_relevant.to_numpy()
    )

    first_relevant_rank = (
        int(
            positions[0] + 1
        )
        if len(positions) > 0
        else None
    )

    return {
        "eval_id":
            row["eval_id"],

        "question_source":
            row[
                "question_source"
            ],

        "first_relevant_rank":
            first_relevant_rank,

        "hit_at_1": (
            first_relevant_rank
            is not None
            and
            first_relevant_rank <= 1
        ),

        "hit_at_3": (
            first_relevant_rank
            is not None
            and
            first_relevant_rank <= 3
        ),

        "hit_at_5": (
            first_relevant_rank
            is not None
            and
            first_relevant_rank <= 5
        ),

        "rr_at_10": (
            1 / first_relevant_rank
            if first_relevant_rank
            is not None
            else 0.0
        ),

        "top1_document_id":
            results.iloc[0][
                "document_id"
            ],

        "top1_section_path":
            results.iloc[0][
                "section_path"
            ],

        "top1_score":
            float(
                results.iloc[0][
                    "score"
                ]
            ),
    }

In [32]:
faiss_eval_results = []

for _, row in (
    retrieval_eval_df.iterrows()
):
    faiss_eval_results.append(
        evaluate_faiss_query(
            row,
            top_k=10,
        )
    )

faiss_eval_results_df = (
    pd.DataFrame(
        faiss_eval_results
    )
)

faiss_eval_results_df.head()

,eval_id,question_source,first_relevant_rank,hit_at_1,hit_at_3,hit_at_5,rr_at_10,top1_document_id,top1_section_path,top1_score
0,dev_0005,real_dev,8.0,False,False,False,0.125,va_dod_major_depression_2022,II. Background > A. Description of Major Depressive Disorder (MDD),0.628073
1,dev_0014,real_dev,1.0,True,True,True,1.000,va_dod_low_back_pain_2022,IX. Recommendations > A. Evaluation and Diagnostic Approach,0.668335
2,dev_0029,real_dev,1.0,True,True,True,1.000,va_dod_ckd_2025,IX. Recommendations,0.723086
3,dev_0048,real_dev,1.0,True,True,True,1.000,va_dod_low_back_pain_2022,IX. Recommendations > D. Pharmacotherapy,0.692542
4,dev_0078,real_dev,2.0,False,True,True,0.500,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,0.646642


In [33]:
faiss_overall_metrics = (
    summarize_retrieval_metrics(
        faiss_eval_results_df
    )
)

faiss_overall_metrics

n         47.000000
Hit@1      0.829787
Hit@3      0.936170
Hit@5      0.936170
MRR@10     0.882092
dtype: float64

In [34]:
faiss_doc_hit_at_1 = (
    faiss_eval_results_df[
        "top1_document_id"
    ].to_numpy()
    ==
    retrieval_eval_df[
        "relevant_document_id"
    ].to_numpy()
).mean()

print(
    "FAISS Document Hit@1:",
    faiss_doc_hit_at_1,
)

FAISS Document Hit@1: 1.0


In [35]:
dense_faiss_comparison_df = pd.DataFrame(
    [
        {
            "method":
                "BGE dense — NumPy",

            "Hit@1":
                dense_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                dense_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                dense_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                dense_eval_results_df[
                    "rr_at_10"
                ].mean(),
        },
        {
            "method":
                "BGE dense — FAISS",

            "Hit@1":
                faiss_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                faiss_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                faiss_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                faiss_eval_results_df[
                    "rr_at_10"
                ].mean(),
        },
    ]
)

dense_faiss_comparison_df[
    [
        "Hit@1",
        "Hit@3",
        "Hit@5",
        "MRR@10",
    ]
] = (
    dense_faiss_comparison_df[
        [
            "Hit@1",
            "Hit@3",
            "Hit@5",
            "MRR@10",
        ]
    ]
    .round(3)
)

dense_faiss_comparison_df

,method,Hit@1,Hit@3,Hit@5,MRR@10
0,BGE dense — NumPy,0.83,0.936,0.936,0.882
1,BGE dense — FAISS,0.83,0.936,0.936,0.882


### Проверка совпадения результатов NumPy и FAISS

Одних агрегированных метрик недостаточно.

Дополнительно проверяем, совпадает ли порядок `top-10` chunks
для каждого из 47 benchmark-запросов.

In [36]:
dense_faiss_agreement_rows = []

for _, row in (
    retrieval_eval_df.iterrows()
):
    dense_results = dense_search(
        query=row["query"],
        chunks_df=chunks_df,
        document_embeddings=(
            document_embeddings
        ),
        embedding_model=(
            embedding_model
        ),
        query_prefix=(
            retrieval_config[
                "query_prefix"
            ]
        ),
        top_k=10,
    )

    faiss_results = faiss_search(
        row["query"],
        top_k=10,
    )

    dense_chunk_ids = (
        dense_results[
            "chunk_id"
        ]
        .astype(str)
        .tolist()
    )

    faiss_chunk_ids = (
        faiss_results[
            "chunk_id"
        ]
        .astype(str)
        .tolist()
    )

    dense_faiss_agreement_rows.append(
        {
            "eval_id":
                row["eval_id"],

            "одинаковый_порядок_top10":
                dense_chunk_ids
                == faiss_chunk_ids,

            "одинаковый_набор_top10":
                set(
                    dense_chunk_ids
                )
                ==
                set(
                    faiss_chunk_ids
                ),
        }
    )

dense_faiss_agreement_df = (
    pd.DataFrame(
        dense_faiss_agreement_rows
    )
)

print(
    "Полное совпадение порядка top-10:",
    dense_faiss_agreement_df[
        "одинаковый_порядок_top10"
    ].mean(),
)

print(
    "Совпадение набора top-10:",
    dense_faiss_agreement_df[
        "одинаковый_набор_top10"
    ].mean(),
)

Полное совпадение порядка top-10: 1.0
Совпадение набора top-10: 1.0


### Сравнение времени только для векторного поиска

Чтобы не смешивать время построения query embedding со временем поиска,
сначала один раз рассчитываем embeddings всех benchmark-запросов.

Затем отдельно измеряем:

- текущий NumPy-поиск;
- FAISS `IndexFlatIP`.

На небольшой knowledge base FAISS не обязан быть быстрее.
Смысл этого эксперимента — показать различие поисковых механизмов
и подготовить архитектуру к более крупному индексу.

In [37]:
import time

benchmark_query_embeddings = (
    embedding_model.encode(
        [
            (
                retrieval_config[
                    "query_prefix"
                ]
                + query_text
            )
            for query_text
            in retrieval_eval_df[
                "query"
            ]
        ],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
)

benchmark_query_embeddings = (
    np.ascontiguousarray(
        benchmark_query_embeddings.astype(
            np.float32
        )
    )
)

In [38]:
TOP_K_SPEED = 10
N_SPEED_REPEATS = 20

numpy_times = []
faiss_times = []

for _ in range(
    N_SPEED_REPEATS
):
    start = time.perf_counter()

    for query_embedding in (
        benchmark_query_embeddings
    ):
        scores = (
            document_embeddings
            @ query_embedding
        )

        np.argsort(
            scores
        )[::-1][
            :TOP_K_SPEED
        ]

    numpy_times.append(
        time.perf_counter()
        - start
    )

    start = time.perf_counter()

    faiss_index.search(
        benchmark_query_embeddings,
        TOP_K_SPEED,
    )

    faiss_times.append(
        time.perf_counter()
        - start
    )

speed_comparison_df = pd.DataFrame(
    [
        {
            "метод":
                "NumPy",

            "медиана_секунд":
                np.median(
                    numpy_times
                ),
        },
        {
            "метод":
                "FAISS IndexFlatIP",

            "медиана_секунд":
                np.median(
                    faiss_times
                ),
        },
    ]
)

speed_comparison_df[
    "медиана_секунд"
] = (
    speed_comparison_df[
        "медиана_секунд"
    ]
    .round(6)
)

speed_comparison_df

,метод,медиана_секунд
0,NumPy,0.007449
1,FAISS IndexFlatIP,0.001363


## 8. Повторное ранжирование медицинской моделью MedCPT

Ранее был проверен общий Cross-Encoder
`cross-encoder/ms-marco-MiniLM-L6-v2`.

На текущем retrieval benchmark он ухудшил все основные метрики:

- Hit@1: 0.830 → 0.638;
- Hit@3: 0.936 → 0.872;
- Hit@5: 0.936 → 0.894;
- MRR@10: 0.882 → 0.756.

Поэтому этот reranker удаляется из рабочего retrieval pipeline.

Однако отрицательный результат не доказывает, что повторное ранжирование
бесполезно в принципе. Возможная причина — несоответствие общей модели
медицинскому домену.

Проверим `ncbi/MedCPT-Cross-Encoder`.

Эта модель обучена для биомедицинского информационного поиска.

Схема эксперимента:

query
↓
BGE embedding
↓
FAISS top-10
↓
MedCPT Cross-Encoder
↓
повторное ранжирование
↓
top-k.

На этом этапе модель только меняет порядок уже найденных кандидатов.
Порог отказа по-прежнему не подбирается.

In [39]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

In [40]:
MEDCPT_MODEL_NAME = (
    "ncbi/MedCPT-Cross-Encoder"
)

MEDCPT_CANDIDATES = 10

medcpt_tokenizer = (
    AutoTokenizer.from_pretrained(
        MEDCPT_MODEL_NAME
    )
)

medcpt_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MEDCPT_MODEL_NAME
    )
)

medcpt_model = (
    medcpt_model.to(
        device
    )
)

medcpt_model.eval()

print(
    "Модель повторного ранжирования:",
    MEDCPT_MODEL_NAME,
)

print(
    "Устройство:",
    device,
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18274.84it/s]


Модель повторного ранжирования: ncbi/MedCPT-Cross-Encoder
Устройство: cuda


In [41]:
def medcpt_scores(
    query,
    texts,
    batch_size=16,
):
    scores = []

    for start in range(
        0,
        len(texts),
        batch_size,
    ):
        batch_texts = texts[
            start:
            start + batch_size
        ]

        pairs = [
            [
                query,
                text,
            ]
            for text
            in batch_texts
        ]

        encoded = (
            medcpt_tokenizer(
                pairs,
                truncation=True,
                padding=True,
                return_tensors="pt",
                max_length=512,
            )
        )

        encoded = {
            key: value.to(
                device
            )
            for key, value
            in encoded.items()
        }

        with torch.no_grad():
            logits = (
                medcpt_model(
                    **encoded
                )
                .logits
                .squeeze(-1)
            )

        scores.extend(
            logits
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    return np.asarray(
        scores,
        dtype=np.float32,
    )

In [42]:
def faiss_search_with_medcpt(
    query,
    candidate_k=10,
    final_k=10,
):
    candidates = faiss_search(
        query,
        top_k=candidate_k,
    ).copy()

    reranker_scores = (
        medcpt_scores(
            query=query,
            texts=(
                candidates[
                    "text"
                ]
                .astype(str)
                .tolist()
            ),
        )
    )

    candidates[
        "dense_score"
    ] = candidates[
        "score"
    ]

    candidates[
        "medcpt_score"
    ] = reranker_scores

    candidates = (
        candidates
        .sort_values(
            "medcpt_score",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    return candidates.head(
        final_k
    )

In [43]:
medcpt_example = (
    faiss_search_with_medcpt(
        query=query,
        candidate_k=10,
        final_k=5,
    )
)

display(
    medcpt_example[
        [
            "dense_score",
            "medcpt_score",
            "document_id",
            "section_path",
            "text",
        ]
    ]
)

,dense_score,medcpt_score,document_id,section_path,text
0,0.789073,15.708791,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,"The aggressiveness of treatment of hyperkalemia depends on the degree of elevation and the presence or absence of electrocardiogram (EKG) findings. Observationally, the risk of death from a given level of hyperkalemia is lower in more advanced CKD, suggesting that there are adaptive mechanisms t..."
1,0.752758,14.976225,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > D. Dietary Considerations,"People with CKD and hyperkalemia are commonly advised to follow low-potassium diets. However, randomized evidence about whether this approach is effective is lacking.(277) An unintended consequence of this advice may be a shift toward less healthful diets. In the early stages of CKD, a high inta..."
2,0.798135,13.860844,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > D. Dietary Considerations,"for the Evaluation and Management of Chronic Kidney Disease (3) recommends, “Provide advice to limit the intake of foods rich in bioavailable potassium (e.g., processed foods) for people with CKD G3-G5 who have a history of hyperkalemia.” The involvement of a renal dietitian can be helpful, as c..."
3,0.768300,13.811989,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > B. Definition,"An acute episode of hyperkalemia is a potassium result above the upper limit of normal that is not known to be chronic. However, there is no consensus on the magnitude, duration, and frequency of elevated potassium values that define chronicity.(277)"
4,0.793694,13.380638,va_dod_ckd_2025,Appendix M. Management of Hyperkalemia > C. Management,upper limit of normal range They acknowledge that it is not known whether EKG changes are sensitive in the prediction of potentially lethal arrhythmia. The KDIGO 2024 Clinical Practice Guideline for the Evaluation and Management of Chronic Kidney Disease (3) recommends the following steps to man...


## 9. Оценка BGE + FAISS + MedCPT

MedCPT оценивается на том же frozen retrieval benchmark.

Это позволяет проверить только вклад повторного ранжирования:

BGE + FAISS

против

BGE + FAISS + MedCPT.

Если MedCPT не улучшает retrieval metrics, он не включается
в финальную архитектуру.

In [44]:
def evaluate_medcpt_query(
    row,
    candidate_k=10,
):
    results = (
        faiss_search_with_medcpt(
            query=row["query"],
            candidate_k=(
                candidate_k
            ),
            final_k=(
                candidate_k
            ),
        )
    )

    gold_document = (
        row[
            "relevant_document_id"
        ]
    )

    gold_sections = set(
        row[
            "relevant_sections"
        ]
    )

    is_relevant = (
        results[
            "document_id"
        ].eq(
            gold_document
        )
        &
        results[
            "section_path"
        ].isin(
            gold_sections
        )
    )

    positions = np.flatnonzero(
        is_relevant.to_numpy()
    )

    first_relevant_rank = (
        int(
            positions[0] + 1
        )
        if len(positions) > 0
        else None
    )

    return {
        "eval_id":
            row["eval_id"],

        "question_source":
            row[
                "question_source"
            ],

        "first_relevant_rank":
            first_relevant_rank,

        "hit_at_1": (
            first_relevant_rank
            is not None
            and
            first_relevant_rank <= 1
        ),

        "hit_at_3": (
            first_relevant_rank
            is not None
            and
            first_relevant_rank <= 3
        ),

        "hit_at_5": (
            first_relevant_rank
            is not None
            and
            first_relevant_rank <= 5
        ),

        "rr_at_10": (
            1 / first_relevant_rank
            if first_relevant_rank
            is not None
            else 0.0
        ),

        "top1_document_id":
            results.iloc[0][
                "document_id"
            ],

        "top1_section_path":
            results.iloc[0][
                "section_path"
            ],

        "top1_dense_score":
            float(
                results.iloc[0][
                    "dense_score"
                ]
            ),

        "top1_medcpt_score":
            float(
                results.iloc[0][
                    "medcpt_score"
                ]
            ),
    }

In [45]:
medcpt_eval_results = []

for _, row in (
    retrieval_eval_df.iterrows()
):
    medcpt_eval_results.append(
        evaluate_medcpt_query(
            row,
            candidate_k=(
                MEDCPT_CANDIDATES
            ),
        )
    )

medcpt_eval_results_df = (
    pd.DataFrame(
        medcpt_eval_results
    )
)

medcpt_eval_results_df.head()

,eval_id,question_source,first_relevant_rank,hit_at_1,hit_at_3,hit_at_5,rr_at_10,top1_document_id,top1_section_path,top1_dense_score,top1_medcpt_score
0,dev_0005,real_dev,10.0,False,False,False,0.100000,va_dod_major_depression_2022,Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > B. Scoring the PHQ-9 (223),0.626393,4.732449
1,dev_0014,real_dev,2.0,False,True,True,0.500000,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,0.649784,-5.105602
2,dev_0029,real_dev,2.0,False,True,True,0.500000,va_dod_ckd_2025,II. Background > A. Description of Chronic Kidney Disease,0.713823,4.362084
3,dev_0048,real_dev,2.0,False,True,True,0.500000,va_dod_low_back_pain_2022,IX. Recommendations > E. Dietary Supplements,0.667629,-7.083460
4,dev_0078,real_dev,3.0,False,True,True,0.333333,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,0.646642,-4.939043


In [46]:
medcpt_overall_metrics = (
    summarize_retrieval_metrics(
        medcpt_eval_results_df
    )
)

medcpt_overall_metrics

n         47.000000
Hit@1      0.489362
Hit@3      0.893617
Hit@5      0.914894
MRR@10     0.670567
dtype: float64

In [47]:
medcpt_doc_hit_at_1 = (
    medcpt_eval_results_df[
        "top1_document_id"
    ].to_numpy()
    ==
    retrieval_eval_df[
        "relevant_document_id"
    ].to_numpy()
).mean()

print(
    "MedCPT Document Hit@1:",
    medcpt_doc_hit_at_1,
)

MedCPT Document Hit@1: 1.0


In [48]:
medcpt_comparison_df = pd.DataFrame(
    [
        {
            "method":
                "BGE + FAISS",

            "Hit@1":
                faiss_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                faiss_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                faiss_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                faiss_eval_results_df[
                    "rr_at_10"
                ].mean(),
        },
        {
            "method":
                "BGE + FAISS + MedCPT",

            "Hit@1":
                medcpt_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                medcpt_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                medcpt_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                medcpt_eval_results_df[
                    "rr_at_10"
                ].mean(),
        },
    ]
)

medcpt_comparison_df[
    [
        "Hit@1",
        "Hit@3",
        "Hit@5",
        "MRR@10",
    ]
] = (
    medcpt_comparison_df[
        [
            "Hit@1",
            "Hit@3",
            "Hit@5",
            "MRR@10",
        ]
    ]
    .round(3)
)

medcpt_comparison_df

,method,Hit@1,Hit@3,Hit@5,MRR@10
0,BGE + FAISS,0.830,0.936,0.936,0.882
1,BGE + FAISS + MedCPT,0.489,0.894,0.915,0.671


### Анализ изменения ранга после MedCPT

Средние метрики показывают общий результат, но полезно также понять,
на скольких запросах MedCPT:

- поднял релевантный chunk выше;
- не изменил его позицию;
- опустил его ниже.

In [49]:
medcpt_rank_comparison_df = (
    faiss_eval_results_df[
        [
            "eval_id",
            "first_relevant_rank",
        ]
    ]
    .rename(
        columns={
            "first_relevant_rank":
                "faiss_rank"
        }
    )
    .merge(
        medcpt_eval_results_df[
            [
                "eval_id",
                "first_relevant_rank",
            ]
        ].rename(
            columns={
                "first_relevant_rank":
                    "medcpt_rank"
            }
        ),
        on="eval_id",
        how="inner",
        validate="one_to_one",
    )
)

In [50]:
def compare_retrieval_ranks(
    row,
):
    faiss_rank = (
        row["faiss_rank"]
    )

    medcpt_rank = (
        row["medcpt_rank"]
    )

    if (
        pd.isna(
            faiss_rank
        )
        and
        pd.isna(
            medcpt_rank
        )
    ):
        return (
            "не найден обоими"
        )

    if pd.isna(
        faiss_rank
    ):
        return "улучшилось"

    if pd.isna(
        medcpt_rank
    ):
        return "ухудшилось"

    if (
        medcpt_rank
        < faiss_rank
    ):
        return "улучшилось"

    if (
        medcpt_rank
        > faiss_rank
    ):
        return "ухудшилось"

    return "без изменений"


medcpt_rank_comparison_df[
    "изменение_ранга"
] = (
    medcpt_rank_comparison_df.apply(
        compare_retrieval_ranks,
        axis=1,
    )
)

medcpt_rank_comparison_df[
    "изменение_ранга"
].value_counts()

изменение_ранга
без изменений       22
ухудшилось          21
не найден обоими     2
улучшилось           2
Name: count, dtype: int64

## 10. Итоговое сравнение методов retrieval

В итоговую таблицу включаются:

1. TF-IDF;
2. BGE dense с текущим NumPy-поиском;
3. BGE dense с FAISS;
4. BGE + FAISS + MedCPT.

FAISS оценивается отдельно, хотя ожидается совпадение качества с NumPy,
поскольку он использует те же BGE embeddings и ту же меру сходства.

MedCPT остаётся в архитектуре только в том случае, если его вклад
подтверждается retrieval benchmark.

In [51]:
dense_doc_comparison_df = (
    dense_eval_results_df[
        [
            "eval_id",
            "top1_document_id",
        ]
    ]
    .merge(
        retrieval_eval_df[
            [
                "eval_id",
                "relevant_document_id",
            ]
        ],
        on="eval_id",
        how="inner",
        validate="one_to_one",
    )
)

dense_doc_hit_at_1 = (
    dense_doc_comparison_df[
        "top1_document_id"
    ]
    .eq(
        dense_doc_comparison_df[
            "relevant_document_id"
        ]
    )
    .mean()
)

In [52]:
retrieval_method_comparison_extended_df = pd.DataFrame(
    [
        {
            "method":
                "TF-IDF",

            "Hit@1":
                tfidf_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                tfidf_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                tfidf_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                tfidf_eval_results_df[
                    "rr_at_10"
                ].mean(),

            "Document_Hit@1":
                tfidf_doc_hit_at_1,
        },
        {
            "method":
                "BGE dense — NumPy",

            "Hit@1":
                dense_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                dense_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                dense_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                dense_eval_results_df[
                    "rr_at_10"
                ].mean(),

            "Document_Hit@1":
                dense_doc_hit_at_1,
        },
        {
            "method":
                "BGE dense — FAISS",

            "Hit@1":
                faiss_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                faiss_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                faiss_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                faiss_eval_results_df[
                    "rr_at_10"
                ].mean(),

            "Document_Hit@1":
                faiss_doc_hit_at_1,
        },
        {
            "method":
                "BGE + FAISS + MedCPT",

            "Hit@1":
                medcpt_eval_results_df[
                    "hit_at_1"
                ].mean(),

            "Hit@3":
                medcpt_eval_results_df[
                    "hit_at_3"
                ].mean(),

            "Hit@5":
                medcpt_eval_results_df[
                    "hit_at_5"
                ].mean(),

            "MRR@10":
                medcpt_eval_results_df[
                    "rr_at_10"
                ].mean(),

            "Document_Hit@1":
                medcpt_doc_hit_at_1,
        },
    ]
)

retrieval_method_comparison_extended_df[
    [
        "Hit@1",
        "Hit@3",
        "Hit@5",
        "MRR@10",
        "Document_Hit@1",
    ]
] = (
    retrieval_method_comparison_extended_df[
        [
            "Hit@1",
            "Hit@3",
            "Hit@5",
            "MRR@10",
            "Document_Hit@1",
        ]
    ]
    .round(3)
)

retrieval_method_comparison_extended_df

,method,Hit@1,Hit@3,Hit@5,MRR@10,Document_Hit@1
0,TF-IDF,0.340,0.596,0.702,0.509,0.936
1,BGE dense — NumPy,0.830,0.936,0.936,0.882,1.000
2,BGE dense — FAISS,0.830,0.936,0.936,0.882,1.000
3,BGE + FAISS + MedCPT,0.489,0.894,0.915,0.671,1.000


In [53]:
dense_hit_1 = (
    faiss_eval_results_df[
        "hit_at_1"
    ].mean()
)

dense_mrr = (
    faiss_eval_results_df[
        "rr_at_10"
    ].mean()
)

medcpt_hit_1 = (
    medcpt_eval_results_df[
        "hit_at_1"
    ].mean()
)

medcpt_mrr = (
    medcpt_eval_results_df[
        "rr_at_10"
    ].mean()
)

if (
    medcpt_hit_1
    >= dense_hit_1
    and
    medcpt_mrr
    >= dense_mrr
    and
    (
        medcpt_hit_1
        > dense_hit_1
        or
        medcpt_mrr
        > dense_mrr
    )
):
    print(
        "MedCPT улучшает основные метрики. "
        "Его можно рассматривать для следующего этапа."
    )
else:
    print(
        "Устойчивое улучшение MedCPT не подтверждено. "
        "Не включаем его в финальный retrieval pipeline "
        "без дополнительного обоснования."
    )

Устойчивое улучшение MedCPT не подтверждено. Не включаем его в финальный retrieval pipeline без дополнительного обоснования.


In [54]:
faiss_eval_results_df.to_csv(
    RESULTS_DIR
    / "retrieval_faiss_eval_v1.csv",
    index=False,
)

dense_faiss_agreement_df.to_csv(
    RESULTS_DIR
    / "retrieval_numpy_vs_faiss_v1.csv",
    index=False,
)

medcpt_eval_results_df.to_csv(
    RESULTS_DIR
    / "retrieval_medcpt_eval_v1.csv",
    index=False,
)

medcpt_rank_comparison_df.to_csv(
    RESULTS_DIR
    / "retrieval_medcpt_rank_changes_v1.csv",
    index=False,
)

retrieval_method_comparison_extended_df.to_csv(
    RESULTS_DIR
    / "retrieval_method_comparison_extended_v1.csv",
    index=False,
)

print(
    "Результаты FAISS и MedCPT сохранены."
)

Результаты FAISS и MedCPT сохранены.


## 11. Порог релевантности и отказ от ответа

FAISS всегда возвращает ближайшие векторы, даже если ни один chunk
не является действительно релевантным запросу.

Поэтому одного `top-k` недостаточно.

Необходимо отдельно определить, есть ли в knowledge base
достаточно релевантная информация.

Для этого вводится порог по retrieval score:

- если лучший найденный chunk имеет score выше порога,
  retrieval считается достаточно уверенным;
- если score ниже порога,
  система не передаёт случайный контекст генератору.

Порог нельзя выбирать по final test.

Он подбирается на отдельном retrieval development set,
который должен содержать как запросы с ответом в knowledge base,
так и запросы вне покрытия knowledge base.

In [57]:
covered_queries_df = (
    retrieval_eval_df[
        [
            "eval_id",
            "query",
        ]
    ]
    .copy()
)

covered_queries_df[
    "covered"
] = 1

covered_queries_df.head()

,eval_id,query,covered
0,dev_0005,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stre...,1
1,dev_0014,"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to deter...",1
2,dev_0029,"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be...",1
3,dev_0048,hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but ...,1
4,dev_0078,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for an...",1


In [58]:
not_covered_queries = [
    "What is the treatment for acne vulgaris?",
    "How is appendicitis diagnosed?",
    "What medications are used for migraine prevention?",
    "How should psoriasis be treated?",
    "What causes glaucoma?",
    "How is otitis media treated in children?",
    "What are the treatments for endometriosis?",
    "How is Parkinson disease diagnosed?",
    "What causes dental enamel erosion?",
    "How is gallstone disease treated?",
    "What are the treatment options for rheumatoid arthritis?",
    "How is iron deficiency anemia treated?",
    "What medications are used for epilepsy?",
    "How should bacterial conjunctivitis be treated?",
    "What are the symptoms of multiple sclerosis?",
    "How is peptic ulcer disease treated?",
    "What is the treatment for polycystic ovary syndrome?",
    "How is hypothyroidism treated?",
    "What causes kidney stones?",
    "How should acute sinusitis be managed?",
]

In [59]:
pd.set_option(
    "display.max_colwidth",
    None,
)

chunks_df[
    [
        "document_id",
    ]
].drop_duplicates()

,document_id
0,who_hypertension_2021
64,cdc_sti_2021
886,va_dod_asthma_2025
1050,va_dod_ckd_2025
1341,va_dod_low_back_pain_2022
1528,va_dod_major_depression_2022
1717,va_dod_pregnancy_2023
1954,va_dod_type2_diabetes_2023


In [60]:
not_covered_queries_df = pd.DataFrame(
    {
        "eval_id": [
            f"negative_{i:03d}"
            for i in range(
                len(not_covered_queries)
            )
        ],
        "query":
            not_covered_queries,
        "covered":
            0,
    }
)

not_covered_queries_df.head()

,eval_id,query,covered
0,negative_000,What is the treatment for acne vulgaris?,0
1,negative_001,How is appendicitis diagnosed?,0
2,negative_002,What medications are used for migraine prevention?,0
3,negative_003,How should psoriasis be treated?,0
4,negative_004,What causes glaucoma?,0


In [61]:
threshold_benchmark_df = pd.concat(
    [
        covered_queries_df,
        not_covered_queries_df,
    ],
    ignore_index=True,
)

print(
    "Всего запросов:",
    len(threshold_benchmark_df),
)

print()

print(
    threshold_benchmark_df[
        "covered"
    ].value_counts()
)

Всего запросов: 67

covered
1    47
0    20
Name: count, dtype: int64


In [62]:
threshold_rows = []

for _, row in (
    threshold_benchmark_df.iterrows()
):

    results = faiss_search(
        query=row["query"],
        top_k=3,
    )

    scores = (
        results["score"]
        .astype(float)
        .tolist()
    )

    threshold_rows.append(
        {
            "eval_id":
                row["eval_id"],

            "query":
                row["query"],

            "covered":
                row["covered"],

            "top1_score":
                scores[0],

            "top2_score":
                scores[1],

            "top3_score":
                scores[2],

            "mean_top3_score":
                np.mean(
                    scores
                ),
        }
    )

threshold_df = pd.DataFrame(
    threshold_rows
)

threshold_df.head()

,eval_id,query,covered,top1_score,top2_score,top3_score,mean_top3_score
0,dev_0005,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?,1,0.628073,0.626393,0.622863,0.625776
1,dev_0014,"Hi, I am suffering from lower back pain going to my lower right ribs. Ive undergone blood test, urinary test, whole abdominal ultrasound and even kidney xray but found nothing on it. Also lately i feel thirsty all the time. Is there any other kind of test that I need to undergo in order to determine the cause of this?.",1,0.668335,0.661984,0.656332,0.662217
2,dev_0029,"I have been told that I have stage 5 advanced chronic kidney disease, blood pressure perfect cholesterol perfect.Feel great look great, drs are surprised say that considering my blood work I shoulod be deathly sick but Im not at all. I was told that i need dialysis and transplant...shouldnt I be tested further",1,0.723086,0.718721,0.713823,0.718543
3,dev_0048,hi sir iam having a sciatica problem and lumber spine problem for this pain i had taken a electro homeopathy treatment in that doctor had given me a electric shock on my left leg main nerve for 3 times and the pain is gone for few days only after that it came again i had taken lot of pills but no cure now my doctor suggest me duzella 30 mg so how much its useful for me pls suggest me some good medicine iam fearing tht any side affects will come because my marriage is thier in a month and iam having high uric acid 9.0,1,0.692542,0.682166,0.679225,0.684644
4,dev_0078,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable.",1,0.646642,0.644570,0.614048,0.635087


In [55]:
def faiss_search_with_threshold(
    query,
    top_k=3,
    threshold=None,
):
    results = faiss_search(
        query=query,
        top_k=top_k,
    )

    if threshold is None:
        return results

    if (
        results.empty
        or
        results.iloc[0]["score"]
        < threshold
    ):
        return results.iloc[0:0].copy()

    return results

In [63]:
threshold_df.groupby(
    "covered"
)["top1_score"].describe()

,count,mean,std,min,25%,50%,75%,max
covered,,,,,,,,
0,20.0,0.615058,0.040991,0.532971,0.591306,0.615861,0.637877,0.730311
1,47.0,0.749525,0.046431,0.628073,0.724942,0.754878,0.781467,0.836091


In [64]:
threshold_candidates = np.arange(
    0.45,
    0.76,
    0.01,
)

threshold_metrics = []

for threshold in threshold_candidates:

    predicted_covered = (
        threshold_df[
            "top1_score"
        ] >= threshold
    ).astype(int)

    true_covered = (
        threshold_df[
            "covered"
        ].astype(int)
    )

    tp = (
        (predicted_covered == 1)
        & (true_covered == 1)
    ).sum()

    tn = (
        (predicted_covered == 0)
        & (true_covered == 0)
    ).sum()

    fp = (
        (predicted_covered == 1)
        & (true_covered == 0)
    ).sum()

    fn = (
        (predicted_covered == 0)
        & (true_covered == 1)
    ).sum()

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    balanced_accuracy = (
        recall + specificity
    ) / 2

    threshold_metrics.append(
        {
            "threshold":
                threshold,

            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),

            "recall":
                recall,

            "specificity":
                specificity,

            "precision":
                precision,

            "balanced_accuracy":
                balanced_accuracy,
        }
    )

threshold_metrics_df = pd.DataFrame(
    threshold_metrics
)

threshold_metrics_df[
    [
        "threshold",
        "recall",
        "specificity",
        "precision",
        "balanced_accuracy",
    ]
].round(3)

,threshold,recall,specificity,precision,balanced_accuracy
0,0.45,1.000,0.00,0.701,0.500
1,0.46,1.000,0.00,0.701,0.500
2,0.47,1.000,0.00,0.701,0.500
3,0.48,1.000,0.00,0.701,0.500
4,0.49,1.000,0.00,0.701,0.500
5,0.50,1.000,0.00,0.701,0.500
6,0.51,1.000,0.00,0.701,0.500
7,0.52,1.000,0.00,0.701,0.500
8,0.53,1.000,0.00,0.701,0.500
9,0.54,1.000,0.05,0.712,0.525


In [65]:
best_threshold_row = (
    threshold_metrics_df
    .sort_values(
        [
            "balanced_accuracy",
            "specificity",
        ],
        ascending=False,
    )
    .iloc[0]
)

best_threshold_row

threshold             0.650000
TP                   45.000000
TN                   19.000000
FP                    1.000000
FN                    2.000000
recall                0.957447
specificity           0.950000
precision             0.978261
balanced_accuracy     0.953723
Name: 20, dtype: float64

## 12. Анализ ошибок порога релевантности

На текущем development benchmark лучший баланс между сохранением релевантных запросов и отклонением запросов вне покрытия knowledge base наблюдается при пороге `0.65`.

При этом:

- 45 из 47 покрытых запросов корректно пропускаются;
- 2 покрытых запроса ошибочно отклоняются;
- 19 из 20 запросов вне покрытия корректно отклоняются;
- 1 запрос вне покрытия ошибочно считается покрытым.

Перед фиксацией порога необходимо вручную изучить эти ошибки.

Порог `0.65` пока рассматривается как кандидат, а не как универсальная константа для любых данных.

In [66]:
CANDIDATE_THRESHOLD = 0.65

threshold_df[
    "predicted_covered"
] = (
    threshold_df[
        "top1_score"
    ] >= CANDIDATE_THRESHOLD
).astype(int)

threshold_df[
    "error_type"
] = "correct"

threshold_df.loc[
    (
        threshold_df["covered"].eq(0)
        &
        threshold_df[
            "predicted_covered"
        ].eq(1)
    ),
    "error_type",
] = "false_positive"

threshold_df.loc[
    (
        threshold_df["covered"].eq(1)
        &
        threshold_df[
            "predicted_covered"
        ].eq(0)
    ),
    "error_type",
] = "false_negative"

threshold_df[
    "error_type"
].value_counts()

error_type
correct           64
false_negative     2
false_positive     1
Name: count, dtype: int64

In [67]:
false_positive_df = (
    threshold_df[
        threshold_df[
            "error_type"
        ].eq(
            "false_positive"
        )
    ]
    [
        [
            "eval_id",
            "query",
            "top1_score",
            "top2_score",
            "top3_score",
            "mean_top3_score",
        ]
    ]
)

false_positive_df

,eval_id,query,top1_score,top2_score,top3_score,mean_top3_score
60,negative_013,How should bacterial conjunctivitis be treated?,0.730311,0.714842,0.667765,0.704306


In [68]:
false_negative_df = (
    threshold_df[
        threshold_df[
            "error_type"
        ].eq(
            "false_negative"
        )
    ]
    [
        [
            "eval_id",
            "query",
            "top1_score",
            "top2_score",
            "top3_score",
            "mean_top3_score",
        ]
    ]
)

false_negative_df

,eval_id,query,top1_score,top2_score,top3_score,mean_top3_score
0,dev_0005,Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?,0.628073,0.626393,0.622863,0.625776
4,dev_0078,"two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable.",0.646642,0.644570,0.614048,0.635087


In [69]:
error_cases = threshold_df[
    threshold_df[
        "error_type"
    ].ne(
        "correct"
    )
]

for _, row in error_cases.iterrows():

    print("=" * 100)

    print(
        "Тип ошибки:",
        row["error_type"],
    )

    print(
        "Запрос:",
        row["query"],
    )

    print(
        "Ожидаемое покрытие:",
        row["covered"],
    )

    print()

    display(
        faiss_search(
            query=row["query"],
            top_k=3,
        )[
            [
                "score",
                "document_id",
                "section_path",
                "text",
            ]
        ]
    )

Тип ошибки: false_negative
Запрос: Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?
Ожидаемое покрытие: 1



,score,document_id,section_path,text
0,0.628073,va_dod_major_depression_2022,II. Background > A. Description of Major Depressive Disorder (MDD),"the most prevalent and disabling form of depression. In addition to the immediate symptoms of depression, MDD precipitates overall poor quality of life (QoL), decreased productivity, increased obesity and sedentary behavior, and increased risk of mortality from suicide and other causes. Social difficulties potentially emerge from the condition, including stigma, loss of employment, and relationship conflict. Approaches used in the literature to categorize the severity of MDD are discussed in Section IX. Anxiety, posttraumatic stress disorder (PTSD), and substance-related disorders are common co-occurring mental illnesses that may exacerbate existing MDD and complicate treatment. Major depressive disorder also co-occurs with many medical illnesses/conditions like diabetes, hypertension, pulmonary disorders, traumatic brain injury, chronic pain, and congestive heart failure, complicating the treatment of medical disorders and MDD, and increasing morbidity and mortality. Major depressive disorder stems from a combination of genetic, biological, environmental, and psychological factors, and therefore requires a whole person approach to care. For example, trauma, loss of a loved one, a difficult relationship, or any stressful situation may trigger MDD, but the condition may emerge void of a clear trigger. Figure 1. Response to Acute Phases of Treatment"
1,0.626393,va_dod_major_depression_2022,Appendix I: Quick Guide to the Patient Health Questionnaire in Clinical Practice > B. Scoring the PHQ-9 (223),"Table I-3). Note: The diagnoses of MDD requires ruling out a history of a manic episode (Bipolar Disorder) and a physical disorder, medication or other drug as the biological cause of the depressive symptoms. In the context of bereavement or other significant loss, symptoms consistent with a major depression can occur, and the diagnosis of MDD is considered if there is indication the symptoms are distinguished from normal response to loss given the individual’s history, cultural norms, and the context of the loss."
2,0.622863,va_dod_major_depression_2022,IX. Recommendations,"of the cardinal symptoms, and severe MDD had 8 – 9 cardinal symptoms. Regarding chronic depression, also termed persistent depressive disorder (or dysthymia) in DSM-5, symptoms must be present for most days over two years. Typically, symptoms do not remit for greater than two months at a time. Appendix K also describes depression subsets in detail. Topic # Recommendation Strengtha Categoryb 1. We suggest that all patients not currently receiving treatment for depression be screened for depression. Weak for Not reviewed, Amended Screening Reviewed, 2. Weak for New-replaced For patients with MDD, we suggest using a quantitative measure of depression severity in the initial treatment planning and to monitor treatment progress at regular intervals to guide shared treatment decision making. Outcomes Monitoring Strong for Reviewed, Amended 3. For patients with MDD who are being treated in the primary care setting, we recommend the use of collaborative/integrated care models. Neither for Reviewed, New-added 4. For patients with MDD, there is insufficient evidence to recommend for or against the use of a team-based model in specialty mental health care settings. nor against Neither for Treatment Setting Reviewed, New-added 5. For patients with MDD, there is insufficient evidence to conclude that interventions delivered by clinicians using telehealth are either superior or inferior to in-person treatment. nor against Reviewed, 6. Strong for New-replaced We recommend that MDD be treated with either psychotherapy or pharmacotherapy as monotherapy, based on patient preference. Factors including treatment response, severity, and chronicity may lead to other treatment strategies such as augmentation, combination treatment, switching of

Тип ошибки: false_negative
Запрос: two months now into it...started off lower back pain then soreness in hip flexor both legs but more on right. now pain and ackey in both legs ham strings and quads. When driving, 10 min in, severe pain or acke from butt down ham into back of knee. I have to get out and stand up. then good for another 10 min. This is getting unbearable.
Ожидаемое покрытие: 1



,score,document_id,section_path,text
0,0.646642,va_dod_low_back_pain_2022,II. Background > A. Description of Low Back Pain,"LBP has been defined as pain, muscle tension, or stiffness localized below the costal margin and above the inferior gluteal folds with or without leg symptoms.(2) Categorizations defined by duration vary but are often delineated as acute (less than four weeks), subacute (4 – 12 weeks), or chronic (more than 12 weeks) (see the glossary in Appendix F for additional definitions). Anatomical contributors to LBP may be present; however “non-specific” LBP, in which it is not possible to detect a discrete source, is common and occurs in up to 85% of cases.(3) LBP is influenced by the interplay of physical, psychological, social, lifestyle, co-morbid health, and modifiable and non-modifiable health factors. The relative contribution and interaction of these factors is variable, fluctuating, and unique to each individual with LBP and is reflected in the individual’s pain, distress, and coping (behavioral) responses which influence levels of disability.(4) Given the potential for multifactorial contributors to the pain experience, patients with LBP can range from low to high levels of complexity."
1,0.644570,va_dod_low_back_pain_2022,VIII. Algorithm > A. Module A: Initial Evaluation of Low Back Pain,"Sidebar 2: Evaluation for Possible Other Conditionsa Possible Other Conditions Red Flags (e.g., signs, symptoms, history) Suggested Evaluationb • Radicular back pain (e.g., sciatica) • Lower extremity dysesthesia and/or paresthesia None Herniated disc • Severe/progressive lower extremity neurologic deficits • Symptoms present >1 month MRIc None Spinal stenosis • Radicular back pain (e.g., sciatica) • Lower extremity dysesthesia and/or paresthesia • Neurogenic claudication • Older age • Severe/progressive lower extremity neurologic deficits • Symptoms present >1 month MRIc Inflammatory LBP Radiography of pelvis, SI joint, and spine area of interest • Morning stiffness • Improvement with exercise • Alternating buttock pain • Awakening due to LBP during the second part of the night (early morning awakening) • Younger age a These conditions usually do not require urgent diagnostic evaluation b Consider specialty consultation c Some patients may have contraindications to MRI, contrast usually not required Abbreviations: LBP: low back pain; MRI: magnetic resonance imaging; SI: sacroiliac"
2,0.614048,va_dod_low_back_pain_2022,Appendix F: Glossary,"Category Term Definition Acute LBP LBP present for fewer than four weeks. Sometimes grouped with subacute LBP as symptoms present for fewer than 12 weeks. Cauda equina syndrome Compression on nerve roots in the lumbosacral spine, usually due to a massive, centrally herniated disc, or severe lumbar spinal stenosis which can result in urinary retention or incontinence from loss of sphincter function, bilateral motor weakness of the lower extremities, and saddle anesthesia. Chronic LBP LBP present for more than 12 weeks. Herniated disc Herniation of the nucleus pulposus of an intervertebral disc through its fibrous outer covering, which can result in compression of adjacent nerve roots or other structures. Neurogenic claudication Symptoms of leg pain (and occasionally numbness, paresthesia, or weakness) while walking or standing, relieved by sitting or spinal flexion. Associated with spinal stenosis. Non-radicular LBP LBP that typically does not radiate past the knee. Non-specific LBP LBP without signs of a serious underlying condition (such as cancer, infection, or CES), spinal stenosis or radiculopathy, or another specific spinal cause (such as vertebral compression fracture or ankylosing spondylitis). Degenerative changes on lumbar imaging are usually considered nonspecific, as they correlate poorly with symptoms. Often called uncomplicated LBP. Progressive neurologic deficit Abnormal finding of altered function attributable to pathology of the nerves, spinal cord, or brain which sh

Тип ошибки: false_positive
Запрос: How should bacterial conjunctivitis be treated?
Ожидаемое покрытие: 0



,score,document_id,section_path,text
0,0.730311,cdc_sti_2021,Gonococcal Infections > Gonococcal Infection Among Adolescents and Adults > Gonococcal Conjunctivitis,Recommended Regimen for Gonococcal Conjunctivitis Among Adolescents and Adults Ceftriaxone 1 g IM in a single dose Providers should consider one-time lavage of the infected eye with saline solution.
1,0.714842,cdc_sti_2021,Gonococcal Infections > Gonococcal Infection Among Adolescents and Adults > Gonococcal Conjunctivitis,"In the only published study of the treatment regarding gonococcal conjunctivitis among adults, all 12 study participants responded to a single 1-g IM injection of ceftriaxone (898). Because gonococcal conjunctivitis is uncommon and data regarding treatment of gonococcal conjunctivitis among adults are limited, consultation with an infectious disease specialist should be considered."
2,0.667765,cdc_sti_2021,Gonococcal Infections > Gonococcal Infection Among Adolescents and Adults > Gonococcal Conjunctivitis > Management of Sex Partners,"Patients should be instructed to refer their sex partners for evaluation and treatment (see Gonococcal Infections, Management of Sex Partners)."


In [70]:
negative_review_rows = []

negative_cases = (
    threshold_df[
        threshold_df[
            "covered"
        ].eq(0)
    ]
    .sort_values(
        "top1_score",
        ascending=False,
    )
)

for _, row in (
    negative_cases.iterrows()
):
    results = faiss_search(
        query=row["query"],
        top_k=3,
    )

    negative_review_rows.append(
        {
            "eval_id":
                row["eval_id"],

            "query":
                row["query"],

            "top1_score":
                results.iloc[0][
                    "score"
                ],

            "document":
                results.iloc[0][
                    "document_id"
                ],

            "section":
                results.iloc[0][
                    "section_path"
                ],

            "text":
                results.iloc[0][
                    "text"
                ],
        }
    )

negative_review_df = pd.DataFrame(
    negative_review_rows
)

pd.set_option(
    "display.max_colwidth",
    300,
)

negative_review_df

,eval_id,query,top1_score,document,section,text
0,negative_013,How should bacterial conjunctivitis be treated?,0.730311,cdc_sti_2021,Gonococcal Infections > Gonococcal Infection Among Adolescents and Adults > Gonococcal Conjunctivitis,Recommended Regimen for Gonococcal Conjunctivitis Among Adolescents and Adults Ceftriaxone 1 g IM in a single dose Providers should consider one-time lavage of the infected eye with saline solution.
1,negative_001,How is appendicitis diagnosed?,0.646611,cdc_sti_2021,Ectoparasitic Infections > Scabies > Diagnosis,"Scabies diagnosis is made by identifying burrows, mites, eggs, or the mites’ feces from affected areas. Skin scrapings can be examined under the microscope to identify organisms, although"
2,negative_015,How is peptic ulcer disease treated?,0.645433,cdc_sti_2021,"Diseases Characterized by Genital, Anal, or Perianal Ulcers > Chancroid > Treatment","Successful antimicrobial treatment for chancroid cures the infection, resolves the clinical symptoms, and prevents transmission to others. In advanced cases, genital scarring and rectal or urogenital fistulas from suppurative buboes can result despite successful therapy."
3,negative_017,How is hypothyroidism treated?,0.641506,va_dod_major_depression_2022,IX. Recommendations > E. Treatment of MDD that is Severe or has a Partial or Limited Response to Initial Treatment,"increased lithium blood levels. In the military population, the use of mood stabilizers and antipsychotics may trigger the need for a medical evaluation board and fitness for duty evaluation. Liothyronine Liothyronine (synthetic T3) has also been studied as part of augmentation strategies and wa..."
4,negative_003,How should psoriasis be treated?,0.640279,cdc_sti_2021,Ectoparasitic Infections > Scabies > Treatment,"Recommended Regimens for Scabies Permethrin 5% cream applied to all areas of the body from the neck down and washed off after 8–14 hours or Ivermectin 200 ug/kg body weight orally, repeated in 14 days* or Ivermectin 1% lotion applied to all areas of the body from the neck down and washed off aft..."
5,negative_016,What is the treatment for polycystic ovary syndrome?,0.637077,cdc_sti_2021,Human Papillomavirus Infections > Anogenital Warts > Treatment,for complications associated with combination therapy. Treatment regimens are classified as either patient-applied or provider-administered modalities. Patient-applied modalities are preferred by certain persons because they can be administered in the privacy of their home. To ensure that patien...
6,negative_000,What is the treatment for acne vulgaris?,0.634939,cdc_sti_2021,Human Papillomavirus Infections > Anogenital Warts > Treatment,for complications associated with combination therapy. Treatment regimens are classified as either patient-applied or provider-administered modalities. Patient-applied modalities are preferred by certain persons because they can be administered in the privacy of their home. To ensure that patien...
7,negative_018,What causes kidney stones?,0.629217,va_dod_ckd_2025,Appendix H. Management of CKD Table,• No concerns for kidney toxicity so may use as clinically indicated Nuclear medicine contrast Kidney stones • Recommend low-sodium diet and sufficient fluid intake to produce urine output >2.2 L/day • Dietary calcium restriction is not recommended even for calcium stones • Send stones for analy...
8,negative_006,What are the treatments for endometriosis?,0.619604,cdc_sti_2021,Pelvic Inflammatory Disease > Treatment,"PID treatment regimens should provide empiric, broad-spectrum coverage of likely pathogens. Multiple parenteral and oral antimicrobial regimens have been effective in achieving clinical and microbiologic cure in randomized clinical trials with short-term follow-up (1171–1173). However, only a li..."
9,negative_005,How is otitis media treated in children?,0.617755,cdc_sti_2021,Chlamydial Infections > Chlamydial Infection Among Neonates > Infant Pneumonia Caused by C. trachomatis > Trea

In [71]:
threshold_df = threshold_df.rename(
    columns={
        "covered":
            "answerable_from_kb"
    }
)

threshold_benchmark_df = (
    threshold_benchmark_df.rename(
        columns={
            "covered":
                "answerable_from_kb"
        }
    )
)

In [72]:
true_answerable = (
    threshold_df[
        "answerable_from_kb"
    ].astype(int)
)

In [73]:
HARD_NEGATIVE_IDS = {
    "negative_013",  # bacterial → gonococcal conjunctivitis
    "negative_018",  # causes kidney stones → management
    "negative_012",  # epilepsy → pregabalin/gabapentin в другом контексте
    "negative_009",  # gallstone treatment → adverse effect mention
    "negative_017",  # hypothyroidism → T3 augmentation in MDD
    "negative_008",  # enamel erosion → periodontal disease
}

In [74]:
threshold_df[
    "negative_type"
] = np.where(
    threshold_df[
        "eval_id"
    ].isin(
        HARD_NEGATIVE_IDS
    ),
    "hard_negative",
    "regular",
)

threshold_df.loc[
    threshold_df[
        "answerable_from_kb"
    ].eq(1),
    "negative_type",
] = "positive"

In [75]:
threshold_df.groupby(
    "negative_type"
)[
    "top1_score"
].describe().round(3)

,count,mean,std,min,25%,50%,75%,max
negative_type,,,,,,,,
hard_negative,6.0,0.622,0.065,0.533,0.596,0.617,0.638,0.730
positive,47.0,0.750,0.046,0.628,0.725,0.755,0.781,0.836
regular,14.0,0.612,0.028,0.559,0.591,0.616,0.637,0.647


### Вывод по анализу порога релевантности

Порог по cosine similarity способен хорошо отфильтровывать явно нерелевантные результаты, однако не решает задачу определения достаточности evidence полностью.

При пороге 0.65 на development benchmark были получены:

- recall = 0.957;
- specificity = 0.950;
- precision = 0.978;
- balanced accuracy = 0.954.

Анализ ошибок показал два разных ограничения.

Во-первых, некоторые действительно полезные фрагменты имеют score ниже 0.65. Например, релевантные результаты для вопросов о депрессивных симптомах после утраты и о боли в пояснице с иррадиацией получили scores 0.628 и 0.647.

Во-вторых, высокий similarity score не гарантирует достаточность найденной информации. Для общего вопроса о лечении бактериального конъюнктивита был найден узкоспециализированный раздел о гонококковом конъюнктивите со score 0.730.

Следовательно, similarity threshold может использоваться как первый фильтр явно слабого retrieval, но не как единственный критерий допуска RAG-контекста.

Для окончательного решения необходима отдельная проверка достаточности evidence для конкретного вопроса.

In [76]:
threshold_df.to_csv(
    RESULTS_DIR
    / "retrieval_answerability_scores_v1.csv",
    index=False,
)

threshold_metrics_df.to_csv(
    RESULTS_DIR
    / "retrieval_threshold_metrics_v1.csv",
    index=False,
)

print(
    "Результаты анализа порога сохранены."
)

Результаты анализа порога сохранены.
